![logo](https://raw.githubusercontent.com/sciknoworg/OntoAligner/main/images/logo-with-background.png)

[![PyPI version](https://badge.fury.io/py/OntoAligner.svg)](https://badge.fury.io/py/OntoAligner)
[![PyPI Downloads](https://static.pepy.tech/badge/ontoaligner)](https://pepy.tech/projects/ontoaligner)
![License](https://img.shields.io/badge/License-Apache%202.0-blue.svg)
[![pre-commit](https://img.shields.io/badge/pre--commit-enabled-brightgreen?logo=pre-commit)](https://github.com/pre-commit/pre-commit)
[![Documentation Status](https://readthedocs.org/projects/ontoaligner/badge/?version=main)](https://ontoaligner.readthedocs.io/)
[![Maintenance](https://img.shields.io/badge/Maintained%3F-yes-green.svg)](MAINTANANCE.md)
 [![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.14533133.svg)](https://doi.org/10.5281/zenodo.14533133)

- **Documentation website**: [https://ontoaligner.readthedocs.io/index.html](https://ontoaligner.readthedocs.io/index.html)
- **Resource Paper**: [https://doi.org/10.1007/978-3-031-94578-6_10](https://doi.org/10.1007/978-3-031-94578-6_10)

--------


# Nested Ensemble learning Aligners in OntoAligner

This notebook demonstrates how to build a nested ensemble alignment workflow in OntoAligner.

Ontology alignment can benefit from more than one type of signal. Some aligners are good at retrieving broad candidate mappings, some are better at refining those candidates, and others use structural or language-model reasoning. Instead of choosing only one strategy, this notebook shows how these strategies can be grouped and combined.

The workflow uses [AlignerPipeline](https://ontoaligner.readthedocs.io/developerguide/pipeline.html)  as the standard unit for running each aligner. Related aligners are grouped with [EnsembleLearningAligner](https://ontoaligner.readthedocs.io/aligner/ensemble_learning.html), and those group-level ensembles are combined again into one final nested ensemble.

The flow below shows how the individual `AlignerPipeline` objects are grouped into ensemble aligners and combined into the final nested ensemble:

---
```text
Mouse-Human dataset
        │
        ├─ llm_pipeline ────────┐
        ├─ rag_pipeline ────────┼─ llm_ensemble ────────────┐
        └─ fsrag_pipeline ──────┘                           │
                                                            │
        ├─ lightweight_pipeline ─┐                          │
        ├─ tfidf_pipeline ───────┼─ retrieval_ensemble ─────┼─ nested_ensemble
        └─ sbert_pipeline ───────┘                          │         |
                                                            │         |
        ├─ sbert_reranking_pipeline ─┐                      │         |
        ├─ tfidf_reranking_pipeline ─┼─ reranking_ensemble ─┘         |
        └─ graph_reranking_pipeline ─┘                                |
                                                                      ↓
                                                              final_matchings
                                                                      │
                                                                      ↓
                                                              evaluation report
                                                                      │
                                                                      ↓
                                                              XML and JSON export
```

---
Contents of this tutorial:

1. Setup and configuration
2. Dataset loading
3. Ensemble construction
4. Nested ensemble execution
5. Evaluation and export

---
## 1️⃣. Setup and Configuration

We begin by importing the OntoAligner modules used throughout the notebook and defining the runtime settings. These settings include ontology paths, model paths, and device configuration.

This setup step keeps the rest of the notebook focused on the alignment workflow rather than repeated configuration.

### Import Libraries

OntoAligner provides separate modules for datasets, encoders, aligners, postprocessors, rerankers, ensembles, and evaluation. We import these components here so they can be used consistently across the different ensemble groups.

In [2]:
# Import necessary libraries
import json
import torch

from sklearn.linear_model import LogisticRegression

# Import necessary modules from the 'ontoaligner' library
# The library provides tools for ontology alignment tasks, including dataset management,
# encoding, retrieval, reranking, evaluation, ensemble voting, and postprocessing.
from ontoaligner.ontology import MouseHumanOMDataset, GraphTripleOMDataset
from ontoaligner.utils import metrics, xmlify
from ontoaligner.encoder import (
    ConceptParentLightweightEncoder,
    ConceptLLMEncoder,
    ConceptParentRAGEncoder,
    ConceptParentFewShotEncoder,
    GraphTripleEncoder,
)
from ontoaligner.aligner import (
    SimpleFuzzySMLightweight,
    TFIDFRetrieval,
    SBERTRetrieval,
    AutoModelDecoderLLM,
    ConceptLLMDataset,
    MistralLLMBERTRetrieverRAG,
    MistralLLMBERTRetrieverFSRAG,
    ConvEAligner,
    CrossEncoderReranking,
)
from ontoaligner.postprocess import (
    TFIDFLabelMapper,
    llm_postprocessor,
    rag_heuristic_postprocessor,
    retriever_postprocessor,
)
from ontoaligner.aligner.ensemble import EnsembleLearningAligner
from ontoaligner.aligner.ensemble.voting import (
    ReciprocalRankFusionVoting,
    ScoreAverageVoting,
)
from ontoaligner import AlignerPipeline

C:\Users\AlluV\Desktop\1\OntoAligner-dev-test\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Define Paths and Runtime Settings

The ontology paths point to the source ontology, target ontology, and reference alignments of Mouse-Human anatomy dataset. The model paths define the retrieval, reranking, and LLM components used later in the notebook.

The runtime device is selected once and reused across the aligners. The LLM path can be changed depending on the available hardware, but the detailed LLM and RAG settings are configured in the next step.

In [3]:
# Define paths for the ontology alignment task
source_ontology_path = "../assets/mouse-human/source.xml"
target_ontology_path = "../assets/mouse-human/target.xml"
reference_matching_path = "../assets/mouse-human/reference.xml"

# Select the runtime device
# CUDA is used when available; otherwise, the notebook runs on CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"

# Define model paths
# The LLM is intentionally small for a runnable full-dataset example.
ir_model_path = "all-MiniLM-L6-v2"
cross_encoder_model_path = "cross-encoder/ms-marco-MiniLM-L6-v2"
llm_model_path = "Qwen/Qwen2.5-0.5B-Instruct"

print("Device:", device)
print("IR model:", ir_model_path)
print("Reranker model:", cross_encoder_model_path)
print("LLM model:", llm_model_path)

Device: cpu
IR model: all-MiniLM-L6-v2
Reranker model: cross-encoder/ms-marco-MiniLM-L6-v2
LLM model: Qwen/Qwen2.5-0.5B-Instruct


### Configure LLM and RAG Components

The RAG-based aligners use both retrieval and language-model generation. In this step, we define the shared configuration values for the retriever, the LLM, and the label mapper used by the LLM-based outputs.

These settings can be adjusted depending on the available hardware. For local runs, smaller LLMs and lower retrieval values keep the notebook easier to execute, while larger models or higher retrieval values can be used for fuller experiments.

In [4]:
# Define a label mapper for LLM outputs
# The mapper maps generated answer text into yes/no alignment labels.
mapper = TFIDFLabelMapper(
    classifier=LogisticRegression(),
    ngram_range=(1, 1),
    label_dict={
        "yes": ["yes", "correct", "true", "same", "equivalent", "valid"],
        "no": ["no", "incorrect", "false", "different", "not same", "invalid"],
    },
)

# Define retrieval configuration for RAG aligners
retriever_config = {
    "device": device,
    "top_k": 5,
    "threshold": 0.1,
}

# Define LLM configuration for RAG aligners
llm_config = {
    "device": device,
    "max_length": 256,
    "max_new_tokens": 10,
    "batch_size": 1,
    "answer_set": {
        "yes": ["yes", "correct", "true", "positive", "valid"],
        "no": ["no", "incorrect", "false", "negative", "invalid"],
    },
}

---
## 2️⃣. Dataset Loading

Before running any aligner, we load the [ontology matching task](https://ontoaligner.readthedocs.io/developerguide/parsers.html). The dataset provides the source ontology, the target ontology, and the reference alignments used for evaluation.

We also load a graph-based version of the dataset because the graph aligner uses ontology structure rather than only concept text.

### Load the Ontology Matching Dataset

This step loads the standard ontology matching dataset. The resulting dataset is used by the retrieval, reranking, and RAG-based aligners.

In [5]:
# Initialize the ontology alignment task
task = MouseHumanOMDataset()
print("Test Task:", task)

# Collect the ontology dataset
dataset = task.collect(
    source_ontology_path=source_ontology_path,
    target_ontology_path=target_ontology_path,
    reference_matching_path=reference_matching_path,
)

print("Dataset keys:", dataset.keys())
print("Reference matchings:", len(dataset["reference"]))

Test Task: Track: anatomy, Source-Target sets: mouse-human


2744it [00:00, 8652.67it/s]
3304it [00:00, 5978.74it/s]
100%|██████████| 9102/9102 [00:00<00:00, 64871.94it/s]

Dataset keys: dict_keys(['dataset-info', 'source', 'target', 'reference'])
Reference matchings: 1516


### Load the Graph Dataset

The graph dataset prepares the same ontology matching task for graph-based alignment. This allows the graph aligner to use structural information from the ontologies.

In [6]:
# Initialize the graph ontology alignment task
# GraphTripleOMDataset prepares the ontology alignment task for graph-based aligners.
graph_task = GraphTripleOMDataset(ontology_name="mouse-human")
print("Graph Task:", graph_task)

# Collect the graph dataset
graph_dataset = graph_task.collect(
    source_ontology_path=source_ontology_path,
    target_ontology_path=target_ontology_path,
    reference_matching_path=reference_matching_path,
)

print("Graph dataset keys:", graph_dataset.keys())

Graph Task: Track: GraphTriple, Source-Target sets: mouse-human


100%|██████████| 9102/9102 [00:00<00:00, 62775.31it/s]

Graph dataset keys: dict_keys(['dataset-info', 'source', 'target', 'reference'])


---
## 3️⃣. Ensemble Construction

In this section, we create the group-level [ensembles](https://ontoaligner.readthedocs.io/aligner/ensemble_learning.html). Each group focuses on a different alignment strategy.

The retrieval ensemble captures lexical and semantic similarity. The reranking ensemble refines candidate mappings using a stronger scoring model. The LLM-based ensemble shows how language-model reasoning can be combined with retrieval.

Each aligner is wrapped with [AlignerPipeline](https://ontoaligner.readthedocs.io/developerguide/pipeline.html), so all groups follow the same execution style. Each ensemble group also defines a [voting strategy](https://ontoaligner.readthedocs.io/aligner/ensemble_learning.html#voting-strategies), which controls how the predictions from its aligners are combined.

### Build the Retrieval Ensemble

We first build the retrieval ensemble. [Retrieval aligners](https://ontoaligner.readthedocs.io/aligner/retriever.html#) are useful because they can quickly generate candidate mappings between source and target concepts.

This group combines lightweight matching, TF-IDF retrieval, and SBERT retrieval. These aligners provide different lexical and semantic views of the same ontology matching task. Reciprocal rank fusion is used to combine retrieval aligners by rank.

In [7]:
# Define the lightweight fuzzy matching pipeline
lightweight_pipeline = AlignerPipeline(
    encoder=ConceptParentLightweightEncoder(),
    aligner=SimpleFuzzySMLightweight(fuzzy_sm_threshold=0.2),
    om_dataset=dataset,
)

# Define the TF-IDF retrieval pipeline
tfidf_pipeline = AlignerPipeline(
    encoder=ConceptParentLightweightEncoder(),
    aligner=TFIDFRetrieval(top_k=5),
    om_dataset=dataset,
    load_params={"path": None},
)

# Define the SBERT retrieval pipeline
sbert_pipeline = AlignerPipeline(
    encoder=ConceptParentLightweightEncoder(),
    aligner=SBERTRetrieval(device=device, top_k=5),
    om_dataset=dataset,
    load_params={"path": ir_model_path},
)

# Combine the retrieval aligners into one ensemble aligner
retrieval_ensemble = EnsembleLearningAligner(
    aligners=[
        ("lightweight", lightweight_pipeline, 1.0),
        ("tfidf", tfidf_pipeline, 1.0),
        ("sbert", sbert_pipeline, 1.0),
    ],
    voting=ReciprocalRankFusionVoting(k=60),
)

### Build the Reranking Ensemble

Next, we build the reranking ensemble. [Reranking](https://ontoaligner.readthedocs.io/aligner/retriever.html#reranking) starts with candidate mappings and then applies a stronger relevance model to reorder or filter those candidates.

In this notebook, reranking is handled directly inside `AlignerPipeline`. This keeps the workflow consistent: the pipeline runs the encoder, aligner, optional reranker, and postprocessor in one place.

The reranking group includes SBERT-based candidates, TF-IDF-based candidates, and graph-based candidates. Score averaging is used because the reranking scores are normalized.

In [8]:
# Define the SBERT reranking pipeline
# SBERT generates candidates, CrossEncoderReranking reranks them, and retriever_postprocessor flattens the output.
sbert_reranking_pipeline = AlignerPipeline(
    encoder=ConceptParentLightweightEncoder(),
    aligner=SBERTRetrieval(device=device, top_k=10),
    reranker=CrossEncoderReranking(
        device=device,
        top_k=5,
        normalize_score="sigmoid",
    ),
    om_dataset=dataset,
    load_params={"path": ir_model_path},
    reranker_load_params={"path": cross_encoder_model_path},
    postprocessor=retriever_postprocessor,
    postprocessor_params={"threshold": 0.5},
)

# Define the TF-IDF reranking pipeline
# TF-IDF provides lexical candidates before the same CrossEncoder reranking step.
tfidf_reranking_pipeline = AlignerPipeline(
    encoder=ConceptParentLightweightEncoder(),
    aligner=TFIDFRetrieval(top_k=10),
    reranker=CrossEncoderReranking(
        device=device,
        top_k=5,
        normalize_score="sigmoid",
    ),
    om_dataset=dataset,
    load_params={"path": None},
    reranker_load_params={"path": cross_encoder_model_path},
    postprocessor=retriever_postprocessor,
    postprocessor_params={"threshold": 0.5},
)

# Define the graph reranking pipeline
# The graph aligner is configured with retriever=True so it returns grouped candidates.
# The reranker_encoder prepares text representations for CrossEncoderReranking.
graph_reranking_pipeline = AlignerPipeline(
    encoder=GraphTripleEncoder(),
    aligner=ConvEAligner(
        model="ConvE",
        device=device,
        retriever=True,
        top_k=10,
        embedding_dim=32,
        num_epochs=1,
        train_batch_size=32,
        eval_batch_size=32,
        num_negs_per_pos=1,
        random_seed=42,
    ),
    reranker=CrossEncoderReranking(
        device=device,
        top_k=5,
        normalize_score="sigmoid",
    ),
    reranker_encoder=ConceptParentLightweightEncoder(),
    reranker_om_dataset=dataset,
    om_dataset=graph_dataset,
    reranker_load_params={"path": cross_encoder_model_path},
    postprocessor=retriever_postprocessor,
    postprocessor_params={"threshold": 0.5},
)

# Combine the reranking aligners into one ensemble aligner
# All aligners use the same reranking model and sigmoid score normalization.
reranking_ensemble = EnsembleLearningAligner(
    aligners=[
        ("sbert_reranking", sbert_reranking_pipeline, 1.0),
        ("tfidf_reranking", tfidf_reranking_pipeline, 1.0),
        ("graph_reranking", graph_reranking_pipeline, 1.0),
    ],
    voting=ScoreAverageVoting(),
)

### Build the LLM-Based Ensemble

The LLM-based ensemble shows how language models can be used for ontology alignment.

This group includes a direct [LLM aligner](https://ontoaligner.readthedocs.io/aligner/llm.html) with `AutoModelDecoderLLM`, along with [RAG](https://ontoaligner.readthedocs.io/aligner/rag.html) and [FS-RAG](https://ontoaligner.readthedocs.io/aligner/rag.html#fewshot-rag-aligner) aligners. The direct LLM aligner compares source and target concepts using generation, while RAG and FS-RAG first retrieve candidate targets and then use the LLM to support the alignment decision. Reciprocal rank fusion is used to combine LLM-based aligners by rank.

In [9]:
# Define the direct decoder LLM pipeline
# The pipeline uses ConceptLLMEncoder and ConceptLLMDataset to generate LLM prompts.
llm_pipeline = AlignerPipeline(
    encoder=ConceptLLMEncoder(),
    aligner=AutoModelDecoderLLM(
        device=device,
        max_length=256,
        max_new_tokens=10,
        batch_size=1,
    ),
    om_dataset=dataset,
    llm_dataset_class=ConceptLLMDataset,
    load_params={"path": llm_model_path},
    postprocessor=llm_postprocessor,
    postprocessor_params={
        "mapper": mapper,
        "interested_class": "yes",
    },
)

# Define the RAG pipeline
# RAG first retrieves candidate targets and then uses an LLM for answer generation.
rag_pipeline = AlignerPipeline(
    encoder=ConceptParentRAGEncoder(),
    aligner=MistralLLMBERTRetrieverRAG(
        retriever_config=retriever_config,
        llm_config=llm_config,
    ),
    om_dataset=dataset,
    load_params={
        "llm_path": llm_model_path,
        "ir_path": ir_model_path,
    },
    postprocessor=rag_heuristic_postprocessor,
    postprocessor_params={
        "topk_confidence_ratio": 3,
        "topk_confidence_score": 3,
    },
)

# Define the few-shot RAG pipeline
# Few-shot RAG adds reference-based examples during prompt construction.
fsrag_pipeline = AlignerPipeline(
    encoder=ConceptParentFewShotEncoder(),
    aligner=MistralLLMBERTRetrieverFSRAG(
        positive_ratio=1.0,
        n_shots=1,
        retriever_config=retriever_config,
        llm_config=llm_config,
    ),
    om_dataset=dataset,
    load_params={
        "llm_path": llm_model_path,
        "ir_path": ir_model_path,
    },
    postprocessor=rag_heuristic_postprocessor,
    postprocessor_params={
        "topk_confidence_ratio": 3,
        "topk_confidence_score": 3,
    },
    include_reference=True,
)

# Combine the LLM aligners into one ensemble aligner
llm_ensemble = EnsembleLearningAligner(
    aligners=[
        ("llm", llm_pipeline, 1.0),
        ("rag", rag_pipeline, 1.0),
        ("fsrag", fsrag_pipeline, 1.0),
    ],
    voting=ReciprocalRankFusionVoting(k=60),
)

---
## 4️⃣. Nested Ensemble Execution

After creating the group-level ensembles, we connect them into one final nested ensemble.

At this stage, the retrieval, reranking, and LLM-based ensembles are treated as aligners inside the final [EnsembleLearningAligner](https://ontoaligner.readthedocs.io/aligner/ensemble_learning.html). When the final nested ensemble is executed, it runs each group-level ensemble, collects their predictions, and combines them into one ranked alignment output.

### Build the Final Nested Ensemble

This cell builds the final `EnsembleLearningAligner` using the group-level ensembles as inputs. When `generate()` is called, the nested ensemble runs the retrieval, reranking, and LLM-based groups through this final ensemble structure.

Reciprocal rank fusion is used at the final level because the different groups may use different scoring semantics.

In [9]:
# Initialize the final nested ensemble aligner
# Each group-level ensemble behaves like an aligner because it exposes generate().
nested_ensemble = EnsembleLearningAligner(
    aligners=[
        ("retrieval_ensemble", retrieval_ensemble, 1.0),
        ("reranking_ensemble", reranking_ensemble, 1.0),
    ],
    voting=ReciprocalRankFusionVoting(k=60),
)

# Optional: Full nested ensemble with llm_ensemble
# Uncomment this version for Colab, GPU, or overnight execution.
# nested_ensemble = EnsembleLearningAligner(
#     aligners=[
#         ("retrieval_ensemble", retrieval_ensemble, 1.0),
#         ("reranking_ensemble", reranking_ensemble, 1.0),
#         ("llm_ensemble", llm_ensemble, 1.0),
#     ],
#     voting=ReciprocalRankFusionVoting(k=60),
# )

# Generate final nested ensemble predictions
final_matchings = nested_ensemble.generate()

# Print a small sample of predictions
print("Final nested ensemble matchings:", len(final_matchings))
print(json.dumps(final_matchings[:20], indent=4, ensure_ascii=False))


Running aligner: retrieval_ensemble

Running aligner: lightweight


100%|██████████| 2737/2737 [00:01<00:00, 1404.64it/s]


Finished aligner: lightweight
Predictions before flattening: 2737
Predictions after flattening: 2737

Running aligner: tfidf


2737it [00:14, 192.37it/s]


Finished aligner: tfidf
Predictions before flattening: 2737
Predictions after flattening: 13685

Running aligner: sbert


...

Finished aligner: sbert_reranking
Predictions before flattening: 5300
Predictions after flattening: 5300

Running aligner: tfidf_reranking


2737it [00:09, 274.21it/s]
100%|██████████| 2737/2737 [00:00<00:00, 384789.50it/s]


Finished aligner: tfidf_reranking
Predictions before flattening: 4872
Predictions after flattening: 4872

Running aligner: graph_reranking


INFO:pykeen.triples.triples_factory:Creating inverse triples.
C:\Users\AlluV\Desktop\1\OntoAligner-dev-test\.venv\lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Training epochs on cpu:   0%|          | 0/1 [00:00<?, ?epoch/s]INFO:pykeen.triples.triples_factory:Creating inverse triples.
INFO:pykeen.training.training_loop:Dropping last (incomplete) batch each epoch (1/1105 (0.09%) batches).

Training batches on cpu:   0%|          | 0.00/1.10k [00:00<?, ?batch/s]
Training batches on cpu:   1%|          | 7.00/1.10k [00:00<00:17, 63.8batch/s]
Training batches on cpu:   1%|▏         | 15.0/1.10k [00:00<00:15, 72.4batch/s]
Training batches on cpu:   2%|▏         | 23.0/1.10k [00:00<00:14, 74.8batch/s]
Training batches on cpu:   3%|▎         | 31.0/1.10k [00:00<00:14, 75.3batch/s]
Training batches on cpu:   4%|▎         | 40.0/1.10k [00:00<0

Finished aligner: graph_reranking
Predictions before flattening: 911
Predictions after flattening: 911
Finished aligner: reranking_ensemble
Predictions before flattening: 5744
Predictions after flattening: 5744
Final nested ensemble matchings: 22620
[
    {
        "source": "http://mouse.owl#MA_0001367",
        "target": "http://human.owl#NCI_C52793",
        "score": 0.02900988017658188
    },
    {
        "source": "http://mouse.owl#MA_0000752",
        "target": "http://human.owl#NCI_C49596",
        "score": 0.026190476190476188
    },
    {
        "source": "http://mouse.owl#MA_0000137",
        "target": "http://human.owl#NCI_C12771",
        "score": 0.02574441687344913
    },
    {
        "source": "http://mouse.owl#MA_0001368",
        "target": "http://human.owl#NCI_C52792",
        "score": 0.02548701298701299
    },
    {
        "source": "http://mouse.owl#MA_0001364",
        "target": "http://human.owl#NCI_C52796",
        "score": 0.025481764612199396
    },
    {


---
## 5️⃣. Evaluation and Export

The final nested ensemble produces a ranked list of candidate alignments. We evaluate these predictions against the reference alignments to measure precision, recall, and F-score.

The generated alignments are also saved in XML and JSON formats so they can be reused outside the notebook.


### Evaluate the final nested ensemble

This cell evaluates the final nested ensemble predictions against the Mouse-Human reference matchings.

In [10]:
# Evaluate the final predictions
evaluation = metrics.evaluation_report(
    predicts=final_matchings,
    references=dataset["reference"],
)

# Print the evaluation report
print("\nNested Ensemble Evaluation Report:")
print(json.dumps(evaluation, indent=4))


Nested Ensemble Evaluation Report:
{
    "intersection": 1443,
    "precision": 6.379310344827586,
    "recall": 95.18469656992085,
    "f-score": 11.957242293669209,
    "predictions-len": 22620,
    "reference-len": 1516
}


### Save the alignment outputs

This cell exports the final nested ensemble predictions in XML and JSON format for later inspection or evaluation.

In [11]:
# Convert final matchings to XML alignment format
xml_str = xmlify.xml_alignment_generator(matchings=final_matchings)

# Save the XML output
xml_output_file_path = "nested_ensemble_alignments.xml"
with open(xml_output_file_path, "w", encoding="utf-8") as xml_file:
    xml_file.write(xml_str)

print(f"Saved XML: {xml_output_file_path}")

# Save the JSON output
json_output_file_path = "nested_ensemble_alignments.json"
with open(json_output_file_path, "w", encoding="utf-8") as json_file:
    json.dump(final_matchings, json_file, indent=4, ensure_ascii=False)

print(f"Saved JSON: {json_output_file_path}")

# Save the evaluation output
evaluation_output_file_path = "nested_ensemble_evaluation.json"
with open(evaluation_output_file_path, "w", encoding="utf-8") as json_file:
    json.dump(evaluation, json_file, indent=4, ensure_ascii=False)

print(f"Saved Evaluation: {evaluation_output_file_path}")


Saved XML: nested_ensemble_alignments.xml
Saved JSON: nested_ensemble_alignments.json
Saved Evaluation: nested_ensemble_evaluation.json


### Optional Top-1 Selection

The nested ensemble returns ranked candidate alignments. In some ontology matching settings, a stricter output is useful, where each source concept keeps only its highest-scoring target candidate.

This optional step applies top-1 selection, evaluates the selected alignments, and saves the top-1 output in XML and JSON format.

In [12]:
# Optional: Top-1 candidate selection, evaluation, and saving
# Keep only the highest-scoring target for each source concept.

ranked_matchings = sorted(
    final_matchings,
    key=lambda prediction: prediction.get("score", 0.0),
    reverse=True,
)

top1_matchings = []
seen_sources = set()

for prediction in ranked_matchings:
    if prediction["source"] not in seen_sources:
        top1_matchings.append(prediction)
        seen_sources.add(prediction["source"])

top1_evaluation = metrics.evaluation_report(
    predicts=top1_matchings,
    references=dataset["reference"],
)

print("\nTop-1 Nested Ensemble Evaluation Report:")
print(json.dumps(top1_evaluation, indent=4))
print("Top-1 predictions:", len(top1_matchings))

# Save top-1 XML
with open("nested_ensemble_top1_alignments.xml", "w", encoding="utf-8") as xml_file:
    xml_file.write(xmlify.xml_alignment_generator(matchings=top1_matchings))

# Save top-1 JSON
with open("nested_ensemble_top1_alignments.json", "w", encoding="utf-8") as json_file:
    json.dump(top1_matchings, json_file, indent=4, ensure_ascii=False)

# Save top-1 evaluation
with open("nested_ensemble_top1_evaluation.json", "w", encoding="utf-8") as json_file:
    json.dump(top1_evaluation, json_file, indent=4, ensure_ascii=False)

print("Saved top-1 XML, JSON, and evaluation files.")


Top-1 Nested Ensemble Evaluation Report:
{
    "intersection": 1148,
    "precision": 41.94373401534527,
    "recall": 75.72559366754618,
    "f-score": 53.985422055019995,
    "predictions-len": 2737,
    "reference-len": 1516
}
Top-1 predictions: 2737
Saved top-1 XML, JSON, and evaluation files.


---
# ✅ Key Takeaways

This notebook demonstrates how OntoAligner can combine multiple ontology alignment model families in a single nested ensemble workflow.

The main idea is to treat each alignment strategy as an `AlignerPipeline`, then group related aligners with `EnsembleLearningAligner`. These group-level ensembles can then be combined again into a final nested ensemble.

In this example, the workflow combines:

- **Retrieval aligners**, which generate candidate mappings using lexical and semantic similarity.
- **Reranking aligners**, which first retrieve candidates and then refine them using a stronger reranking model.
- **Graph-based aligners**, which use ontology structure to generate candidate mappings.
- **LLM-based aligners**, which use retrieval and language-model reasoning.

This structure makes the alignment process modular. Any OntoAligner aligner can be used through `AlignerPipeline` as long as it follows the standard pipeline interface. This allows different aligners, encoders, postprocessors, and optional rerankers to be composed in the same workflow.

The nested ensemble shows that OntoAligner can support:

```text
multiple aligners
→ multiple model families
→ multiple ensemble groups
→ one final alignment output
```

Overall, this notebook presents a flexible ensemble-based alignment design where different OntoAligner aligners can work together in one consistent pipeline and ensemble structure.

For more information, visit the [OntoAligner Documentation](https://ontoaligner.readthedocs.io/)

-----------------------------------------------------------
-----------------------------------------------------------

📃 Acknowledgement

OntoAligner is licensed under [![License](https://img.shields.io/badge/License-Apache%202.0-blue.svg)](https://opensource.org/licenses/Apache-2.0)


```bibtex
@inproceedings{babaei2025ontoaligner,
  title={OntoAligner: A Comprehensive Modular and Robust Python Toolkit for Ontology Alignment},
  author={Babaei Giglou, Hamed and D’Souza, Jennifer and Karras, Oliver and Auer, S{\"o}ren},
  booktitle={European Semantic Web Conference},
  pages={174--191},
  year={2025},
  organization={Springer}
}
```